# Module 4: Q&A Retrieval-Augmented Generation (RAG) Pipeline

This notebook indexes customer support instruction/response pairs into a local vector database (ChromaDB) using `sentence-transformers/all-MiniLM-L6-v2`, retrieves top grounded chunks, and formats the tone-adjusted prompt for Groq LLM generation.

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

## 1. Load Knowledge Base Chunks

In [ ]:
dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
df = pd.DataFrame(dataset)
print('Total entries:', len(df))
print('Columns:', df.columns.tolist())

sample_df = df.drop_duplicates(subset=['instruction']).head(1000)
print('Sample size for demonstration:', len(sample_df))

## 2. Ingest into ChromaDB with Sentence Transformers

In [ ]:
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
client = chromadb.Client()
collection = client.create_collection(name='demo_support', metadata={'hnsw:space': 'cosine'})

docs = sample_df['instruction'].tolist()
metas = [{'response': r, 'intent': i, 'category': c} for r, i, c in zip(sample_df['response'], sample_df['intent'], sample_df['category'])]
ids = [f'id_{i}' for i in range(len(docs))]

embeddings = embedder.encode(docs, convert_to_numpy=True).tolist()
collection.add(documents=docs, embeddings=embeddings, metadatas=metas, ids=ids)
print('Added records to Chroma collection:', collection.count())

## 3. Grounded Context Retrieval

In [ ]:
query = 'Can I change my delivery address before the order is dispatched?'
query_vec = embedder.encode(query, convert_to_numpy=True).tolist()

results = collection.query(query_embeddings=[query_vec], n_results=3)

print(f'Query: "{query}"\n')
for i in range(len(results['documents'][0])):
    print(f"--- Result {i+1} ---")
    print('Instruction Reference:', results['documents'][0][i])
    print('Official Response:', results['metadatas'][0][i]['response'])
    print('Category:', results['metadatas'][0][i]['category'])
    print()

## 4. Prompt Assembly with Tone Adaptation

In [ ]:
def build_rag_prompt(user_message, detected_sentiment, retrieved_results):
    context_blocks = ''
    for idx, (doc, meta) in enumerate(zip(retrieved_results['documents'][0], retrieved_results['metadatas'][0]), 1):
        context_blocks += f"Support Entry {idx}:\nReference Query: {doc}\nVerified Policy: {meta['response']}\n\n"
        
    prompt = (
        f"You are a helpful, professional customer support assistant for an online retailer. "
        f"Answer the customer's question using ONLY the information in the retrieved support responses below. "
        f"If the customer sounds frustrated ({detected_sentiment}), acknowledge that before answering. "
        f"If the retrieved context does not cover the question, say so honestly and offer to escalate to a human agent rather than guessing.\n\n"
        f"Context (retrieved past support responses):\n{context_blocks}"
        f"Customer question: \"{user_message}\"\n\n"
        f"Answer:"
    )
    return prompt

prompt = build_rag_prompt(query, 'frustrated', results)
print(prompt)

## 5. Generation with Groq LLM (with Fallback)

In [ ]:
def generate_answer(prompt, top_response, sentiment='neutral'):
    api_key = os.environ.get('GROQ_API_KEY')
    if api_key:
        client = Groq(api_key=api_key)
        completion = client.chat.completions.create(
            messages=[
                {'role': 'system', 'content': 'You are an official customer support representative.'},
                {'role': 'user', 'content': prompt}
            ],
            model='llama-3.3-70b-versatile',
            temperature=0.2
        )
        return completion.choices[0].message.content
    else:
        tone_prefix = 'We understand your frustration and apologize for the inconvenience. ' if sentiment == 'frustrated' else ''
        return f"{tone_prefix}{top_response}"

answer = generate_answer(prompt, results['metadatas'][0][0]['response'], sentiment='frustrated')
print('Generated Response:\n', answer)